# A/B Testing: Statistical Power, Conversion Z-Tests, and Multiple Testing

This notebook demonstrates how to design and run an A/B test in production, how to compute the required sample sizes for a given statistical power, and how to protect against false positives (Type I errors) when testing multiple prompt configurations or variants simultaneously using the **Bonferroni Correction**.

## 1. Import Dependencies and Plot Statistical Power Curves

We use `statsmodels` to compute statistical power. If you don't have it installed, run:
```bash
pip install statsmodels
```

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from scipy.stats import norm

try:
    from statsmodels.stats.power import NormalIndPower
    from statsmodels.stats.multitest import multipletests
    statsmodels_ready = True
except ImportError:
    print("statsmodels is not installed. Using simulated power functions instead.")
    statsmodels_ready = False

# Plot sample size vs power curves
sample_sizes = np.linspace(100, 15000, 200)

if statsmodels_ready:
    power_analysis = NormalIndPower()
    # Standardized effect sizes (Cohen's h) for varying MDE levels
    # Suppose baseline conversion rate is 5%
    h_1pct = 2 * (np.arcsin(np.sqrt(0.06)) - np.arcsin(np.sqrt(0.05)))
    h_2pct = 2 * (np.arcsin(np.sqrt(0.07)) - np.arcsin(np.sqrt(0.05)))
    
    power_1 = [power_analysis.power(effect_size=h_1pct, nobs1=n, ratio=1.0, alpha=0.05) for n in sample_sizes]
    power_2 = [power_analysis.power(effect_size=h_2pct, nobs1=n, ratio=1.0, alpha=0.05) for n in sample_sizes]
else:
    # Simulated curves representing typical power scaling
    power_1 = 1.0 - np.exp(-sample_sizes / 10000.0)
    power_2 = 1.0 - np.exp(-sample_sizes / 3000.0)

plt.figure(figsize=(8, 4.5))
plt.plot(sample_sizes, power_1, color='#f76707', linewidth=2, label='MDE = 1.0% absolute (6% vs 5%)')
plt.plot(sample_sizes, power_2, color='#339af0', linewidth=2.5, label='MDE = 2.0% absolute (7% vs 5%)')
plt.axhline(y=0.8, color='#e03131', linestyle='--', label='Target Power Threshold (80%)')
plt.title("A/B Testing Sample Size vs. Statistical Power")
plt.xlabel("Sample Size per Variant (n)")
plt.ylabel("Statistical Power (1 - β)")
plt.legend()
plt.grid(True, linestyle='--')
plt.show()

## 2. Execute Two-Sample Z-Test for Proportions

Suppose we ran a conversion experiment on a web checkout page:
- **Control (A):** $n_A = 1200$, conversions $x_A = 60$ (Conversion rate: 5%)
- **Treatment (B):** $n_B = 1200$, conversions $x_B = 84$ (Conversion rate: 7%)

In [ ]:
n_A, x_A = 1200, 60
n_B, x_B = 1200, 84

p_A = x_A / n_A
p_B = x_B / n_B

# Pooled conversion rate
p_c = (x_A + x_B) / (n_A + n_B)

# Standard Error
se = np.sqrt(p_c * (1.0 - p_c) * (1.0 / n_A + 1.0 / n_B))

# Compute Z-statistic
z_stat = (p_B - p_A) / se

# Calculate two-tailed p-value
p_val = 2 * (1.0 - norm.cdf(abs(z_stat)))

print("--- Conversion Z-Test Results ---")
print(f"Control Rate: {p_A:.2%}, Treatment Rate: {p_B:.2%}")
print(f"Z-statistic: {z_stat:.4f}")
print(f"p-value: {p_val:.4f}")
print(f"Significant at alpha=0.05: {p_val < 0.05}")

## 3. Simulate Multiple Testing Inflation & Apply Bonferroni Correction

If we test 10 prompt template variants at the same time against a single control, our false alarm rate inflates from 5% to 40%.
Let's simulate 10 raw p-values and apply the Bonferroni correction to verify which variants remain statistically significant.

In [ ]:
# Let's simulate p-values for 10 variants. 
# Two are genuine winners (low p-values), one is a marginal false alarm, and 7 are noise.
raw_p_values = [0.0001, 0.0012, 0.0380, 0.1200, 0.4500, 0.0450, 0.8900, 0.2300, 0.6700, 0.0480]
variants = [f"Variant {i+1}" for i in range(10)]

if statsmodels_ready:
    # Apply Bonferroni correction
    reject, corrected_p_vals, _, _ = multipletests(raw_p_values, alpha=0.05, method='bonferroni')
    
    df_tests = pd.DataFrame({
        'Variant': variants,
        'Raw p-value': raw_p_values,
        'Corrected p-value': corrected_p_vals,
        'Significant (Alpha=0.05)': reject
    })
    
    print("--- Bonferroni Multiple Testing Correction ---")
    print(df_tests.to_string(index=False))
else:
    # Manual Bonferroni Calculation
    k = 10
    alpha_target = 0.05
    alpha_adjusted = alpha_target / k
    
    df_tests = pd.DataFrame({
        'Variant': variants,
        'Raw p-value': raw_p_values,
        'Adjusted Alpha Threshold': alpha_adjusted,
        'Significant': [p < alpha_adjusted for p in raw_p_values]
    })
    
    print("--- Manual Bonferroni Correction ---")
    print(df_tests.to_string(index=False))

### Interpretation
- Under a naive analysis without correction, **Variant 3 (p=0.0380), Variant 6 (p=0.0450), and Variant 10 (p=0.0480)** would have been declared as "statistically significant winners" since their p-values are below $0.05$.
- After Bonferroni correction, these marginal false alarms are rejected. Only **Variant 1** and **Variant 2** are recognized as true statistically significant winners, preventing you from deploying false positives to production.